# Cal3Bundler

Bundler is a popular open-source implementation of bundle adjustment, and GTSAM implements its calibration model with radial distortion. `Cal3Bundler` uses one focal length and two radial coefficients; its principal point is stored but is not part of the three-dimensional optimization manifold.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/Cal3Bundler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

## Initialization

The explicit constructor makes the model and parameter order visible. A default constructor is also available, but calibrated values are preferable in real projection code.

In [ ]:
calibration = gtsam.Cal3Bundler(
    800.0, 1.0e-3, -2.0e-5, 320.0, 240.0
)
print(calibration)

## Calibration parameters

`f()`, `k1()`, and `k2()` are the optimized parameters; `px()` and `py()` locate the fixed principal point. `vector()` collects the optimized parameters, while `K()` returns the intrinsic matrix associated with the linear part of the model.

In [ ]:
print("f, k1, k2:", calibration.f(), calibration.k1(), calibration.k2())
print("principal point:", calibration.principalPoint())
print("optimized vector:", calibration.vector())

## Calibrating and uncalibrating points

`uncalibrate()` maps normalized image coordinates to pixels and applies this model's distortion. `calibrate()` performs the inverse mapping iteratively. Their round trip is the most direct way to check parameter conventions.

In [ ]:
normalized = np.array([0.12, -0.08])
pixel = calibration.uncalibrate(normalized)
recovered = calibration.calibrate(pixel)
print("pixel:", pixel)
print("recovered normalized point:", recovered)
np.testing.assert_allclose(recovered, normalized, atol=1e-7)

## Manifold operations

Calibration objects participate in nonlinear optimization through `retract()` and `localCoordinates()`. The tangent coordinates follow the order returned by `vector()` for the parameters that this model optimizes.

In [ ]:
delta = np.array([2.0, 1e-5, -1e-6])
perturbed = calibration.retract(delta)
recovered_delta = calibration.localCoordinates(perturbed)
print("parameter increment:", recovered_delta)
np.testing.assert_allclose(recovered_delta, delta, atol=1e-9)

## Source

[`Cal3Bundler.h`](../Cal3Bundler.h)

## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.